# Feedforward neuraal netwerk met _back propagation_

## Leerdoelen
- Feedforward neural network implementeren met PyTorch
- PyTorch nn.Module gebruiken voor model definitie
- ReLU activatie en Softmax output voor multi-class classificatie
- Cross-entropy loss gebruiken met PyTorch
- Model trainen met PyTorch optimizer (SGD)
- Model performance evalueren op 3-class classificatie
- PyTorch's ingebouwde tools gebruiken voor visualisatie en debugging

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import torch
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch import nn, optim

rng = np.random.default_rng(67)
torch.manual_seed(67)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Data Loading en Exploratie

We gebruiken de volledige Iris dataset met **alle 3 klassen**:
- **Setosa** (klasse 0)
- **Versicolor** (klasse 1)
- **Virginica** (klasse 2)

In [ ]:
# Load the complete Iris dataset
iris = load_iris()

# Create a DataFrame
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df["species"] = iris.target
df["species_name"] = df["species"].map({0: "setosa", 1: "versicolor", 2: "virginica"})

print(f"Total samples: {len(df)}")
print("\nClass distribution:")
for i, name in enumerate(["setosa", "versicolor", "virginica"]):
    count = (df["species"] == i).sum()
    print(f"  {name.capitalize()} ({i}): {count} samples ({100 * count / len(df):.1f}%)")

print("\nFeatures: {iris.feature_names}")
print("\nFirst few rows from each class:")
df.groupby("species_name").head(3)

In [ ]:
# Visualize the 3 classes in 2D (using the two most discriminative features)
fig = px.scatter(
    df,
    x="petal length (cm)",
    y="petal width (cm)",
    color="species_name",
    color_discrete_map={"setosa": "#1f77b4", "versicolor": "#e377c2", "virginica": "#17becf"},
    title="Iris Dataset: All 3 Classes (Petal Dimensions)",
    labels={"petal length (cm)": "Petal Length (cm)", "petal width (cm)": "Petal Width (cm)"},
    width=800,
    height=600,
)
fig.update_traces(marker={"size": 10, "line": {"width": 1, "color": "white"}})
fig.show()

## 2. Data Voorbereiding

We bereiden de data voor door:
1. **Train/test split** (80/20)
2. **Feature standardization** (mean=0, std=1)
3. **One-hot encoding** van de target labels voor multi-class classificatie

## 3. Neural Network Architectuur met PyTorch

We bouwen een feedforward neural network met PyTorch's `nn.Module`:

**Input Layer → Hidden Layer → Output Layer**

- **Input**: 4 features (sepal length, sepal width, petal length, petal width)
- **Hidden Layer**: 8 neurons met ReLU activatie
- **Output Layer**: 3 neurons (voor 3 klassen)

### Loss Functie

**CrossEntropyLoss** in PyTorch (incl. _Softmax_):
$$
\mathcal{L}_{CE} = -\frac{1}{n} \sum_{i=1}^{n} \log\left(\frac{e^{z_{y_i}}}{\sum_{j=1}^{K} e^{z_j}}\right)
$$

## 4. Loss Functie en Optimizer

PyTorch maakt training veel eenvoudiger:
- **Loss functie**: `nn.CrossEntropyLoss()` combineert Softmax + Cross-Entropy
- **Optimizer**: `optim.SGD()` implementeert gradient descent met momentum optie
- **Backpropagation**: Gebeurt automatisch met `loss.backward()`

## 5. Training het Neural Network met PyTorch

De training loop is veel eenvoudiger met PyTorch:
1. **Forward pass**: `outputs = model(inputs)`
2. **Compute loss**: `loss = criterion(outputs, targets)`
3. **Backward pass**: `loss.backward()` (automatische backpropagation!)
4. **Update weights**: `optimizer.step()`

## 6. Visualisatie van Training Proces

We visualiseren hoe de loss en accuracy evolueren tijdens training.

## 7. Model Evaluatie

Laten we de voorspellingen analyseren en een confusion matrix maken om te zien hoe goed het model elke klasse herkent.

## 8. Decision Boundaries Visualisatie

We visualiseren de decision boundaries van het neural network in 2D (gebruikmakend van de twee belangrijkste features).

## 9. Model Inspectie met PyTorch

PyTorch biedt handige tools om het model te inspecteren.

In [ ]:
# Model Summary
print("=" * 70)
print("MODEL SUMMARY")
print("=" * 70)
print(f"\n{model}\n")

print("=" * 70)
print("PARAMETER DETAILS")
print("=" * 70)
for name, param in model.named_parameters():
    print(f"\n{name}:")
    print(f"  Shape: {param.shape}")
    print(f"  Number of parameters: {param.numel()}")
    print(f"  Requires gradient: {param.requires_grad}")
    print(f"  Sample values:\n{param.data[:3] if param.dim() > 1 else param.data}")

print(f"\n{'=' * 70}")
print(f"TOTAL PARAMETERS: {sum(p.numel() for p in model.parameters())}")
print(f"TRAINABLE PARAMETERS: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")
print("=" * 70)

# Gradient inspection (after training)
print("\n" + "=" * 70)
print("GRADIENT INFORMATION (from last training step)")
print("=" * 70)
model.train()
outputs = model(X_train_tensor)
loss = criterion(outputs, y_train_tensor)
optimizer.zero_grad()
loss.backward()

for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"\n{name}:")
        print(f"  Gradient shape: {param.grad.shape}")
        print(f"  Gradient mean: {param.grad.mean().item():.6f}")
        print(f"  Gradient std: {param.grad.std().item():.6f}")
        print(f"  Gradient norm: {param.grad.norm().item():.6f}")